In [27]:
import pandas as pd
import requests, sys
import numpy as np

In [26]:
# ENSO

# Nino3.4
# (NOAA ERSST V5)
# https://psl.noaa.gov/data/timeseries/month/DS/Nino34_CPC/
# 1958-last month
# Download link:
# https://psl.noaa.gov/data/correlation/nina34.anom.data
    

#downloaded file will be saved here
outdir="./data/ENSO/"

#data will be written to this file
outfile="{}/nino34_psl.csv".format(outdir)

#url for downloading data
url="https://psl.noaa.gov/data/correlation/nina34.anom.data"

print(f"Downloading from {url}")

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    print("Success!")
except requests.exceptions.ConnectionError:
    print("Error: No internet connection or server unreachable.")
except requests.exceptions.Timeout:
    print("Error: The request timed out.")
except requests.exceptions.RequestException as e:
    # Catches any other requests-related error
    print(f"An error occurred: {e}")

#parsing data

#skipping last rows that do not contain data
data=response.text.split("\n")[1:-4]

#converting data into array
data=np.array([x.split() for x in data])
data=data.astype(float)

#parsing dates
years=data[:,0].astype(int)

#parsing actual data
data=data[:,1:].flatten()

#creating date index
index=pd.date_range("{}-01-01".format(years[0]), "{}-12-31".format(years[-1]), freq="ME")

#converting data into pandas dataframe
nino34=pd.DataFrame(data, index=index, columns=["nino34"])

#some housekeeping - substituting missing data with nan
nino34[nino34==-99.99]=np.nan

#selecting only valid values
nino34=nino34[~np.isnan(nino34).values]

#saving data into csv file
nino34.to_csv(outfile)
print("saved {}".format(outfile))

#feedback
print("most recent value:", nino34.index[-1], nino34.iloc[-1].values[0])


Success!
saved ./data/ENSO//nino34_psl.csv
most recent value: 2026-03-31 00:00:00 -0.06
